# Build Your AI Agent with MCPs using vLLM, Pydantic AI, and AMD MI300X GPU

Welcome to this hands-on workshop! Throughout this tutorial, we'll leverage AMD GPUs and **Model Context Protocol (MCP)** ,an open standard for exposing LLM tools via API, to deploy powerful language models like Qwen3. Key components:
- 🖥️ **vLLM** for GPU-optimized inference
- 🛠️ **Pydantic-AI** for agent/tool management
- 🔌 **MCP Servers** for pre-built tool integration

You'll learn how to set up your environment, deploy large language models like Qwen3, connect them to real-world tools using MCP, and build a conversational agent capable of reasoning and taking actions.

By the end of this workshop, you’ll have built an AI-powered Airbnb assistant agent—one that can find a place to stay based on your preferences like location, budget, and travel dates.

Let’s dive in!

## Table of Contents

- [Step 1: Launching vLLM Server on AMD GPUs](#step1)
- [Step 2: Installing Dependencies](#step2)
- [Step 3: Create a simple instance of Pydantic-AI Agent](#step3)
- [Step 4: Write a Date/Time Tool for Your Agent](#step4)
- [Step 5: Replace Your Date/time Tool with a MCP server](#step5)
- [Step 6: Turn your agent to an Airbnb finder](#step6)
- [Step 7: Challenge](#step7)

<a id="step1"></a>

## Step 1: Launch a vLLM Server

In this workshop we are going to use [vLLM](https://github.com/vllm-project/vllm) as our inference serving engine. vLLM provides many benefits such as fast model execution, extensive list of supported models, easy to use, and best of all it's open-source. 

### Deploy Qwen3-30B-A3B Model with vLLM




Time to start your vLLM server and creating an end-point for your LLM. Let's open a terminal using your Jupyter server. Then run the following command in this terminal to start the vLLM server:

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \
vllm serve Qwen/Qwen3-30B-A3B \
    --served-model-name Qwen3-30B-A3B \
    --api-key abc-123 \
    --port 8000 \
    --enable-auto-tool-choice \
    --tool-call-parser hermes \
    --trust-remote-code
```

Open another terminal and monitor the GPU utilization by running this command:

```bash
watch rocm-smi
```

Upon successful launch, your server should be accepting incoming traffic through an OpenAI-compatible API. Let's set some environment variables for our server so we can use throughout this tutorial:

In [1]:
import os

BASE_URL = f"http://localhost:8000/v1"

os.environ["BASE_URL"]    = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"   

print("Config set:", BASE_URL)

Config set: http://localhost:8000/v1


We can verify your model is available at the `BASE_URL` we just set by running the following command.

In [2]:
!curl http://localhost:8000/v1/models -H "Authorization: Bearer $OPENAI_API_KEY"

curl: (7) Failed to connect to localhost port 8000 after 0 ms: Connection refused


Congratulations, you now just launched a powerful server that can serve any incoming request and allowing you to build amazing applications. Wasn't that easy?🎉 

<a id="step2"></a>

## Step 2: Installing Dependencies

We are going to use `Pydantic AI`. Let's install the dependencies:

In [3]:
!pip install -q pydantic-ai-slim openai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip



<a id="step3"></a>

## Step 3: Create a simple instance of Pydantic-AI Agent

Let's start by creating a custom OpenAI Compatible endpoint for our agent. 


In [4]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

provider = OpenAIProvider(
    base_url=os.environ["BASE_URL"],
    api_key=os.environ["OPENAI_API_KEY"],
)

agent_model = OpenAIChatModel("Qwen3-30B-A3B", provider=provider)

Let's start by creating an instance the `Agent` class from `pydantic_ai`. 


In [7]:
from pydantic_ai import Agent

agent = Agent(
    model=agent_model
)

It's time to test the agent. `pydantic_ai` provides multiple ways to run `Agent`. You can learn more about it [here](https://ai.pydantic.dev/agents/#running-agents).

In this workshop, we are running in `async` mode. We are going to define a helper function that allows us to quickly test our agent throughout this workshop.

In [5]:
import asyncio
from pydantic_ai.mcp import MCPServerStdio
async def run_async(prompt: str) -> str:
    async with agent.run_mcp_servers():
        result = await agent.run(prompt)
        return result.output


Test the agent by calling this function.

In [8]:
await run_async("What is the capital of India?")

'\n\nThe capital of India is **New Delhi**. It is located in the **National Capital Territory of Delhi** and serves as the political and administrative center of the country. New Delhi was established as the capital in 1911, replacing Calcutta (now Kolkata), and is home to key government institutions, including the Parliament of India, the Rashtrapati Bhavan (Presidential Palace), and the Supreme Court. \n\nWhile "Delhi" often refers to the broader metropolitan area, **New Delhi** specifically denotes the planned administrative region within it.'

Great! now that we have the basics of creating an agent instance, and connecting it to the model we started serving with vLLM earlier.

<a id="step4"></a>

## Step 4: Write a Date/Time Tool for Your Agent

LLMs naturally rely on their training data to respond to your prompts. Therefore, the agent we just defined fails to answer a factual question that falls outside of it's training knowledge. Let's show this with an example:

In [9]:
await run_async("What’s the date today?")

'\n\nI don\'t have access to real-time data, so I can\'t provide the current date. However, you can check the date on your device (like a phone, computer, or smartwatch) or search for "today\'s date" in a search engine. Let me know if there\'s anything else I can help with! 😊'

It is no surprise that the model failed to answer this question. Now, it's time to power-up your LLM by providing `agent` a function that can get the current date. The process of an LLM triggering a function call is commonly referred to as `Tool Calling` or `Function Calling`. In this workshop we are going to take advantage of `pydantic-ai`'s agent `tools` to provide our agent appropriate tools. First, we need to define a custom tool. Below is how we can define a tool in this framework.

In [10]:
from datetime import datetime
from pydantic_ai import Tool          
@Tool
def get_current_date() -> str:
    """Return the current date/time as an ISO-formatted string."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


Next, we need to provide this tool to our Agent, as this will notify the LLM about the existence of such a tool we just definied. This is simply done by just providing the function signiture of the tool we just defined to our agent constructor. 

In [11]:
agent = Agent(
    model=agent_model,
    tools=[get_current_date],
    system_prompt = (
        "You have access to:\n"
        "   1. get_current_time(params: dict)\n"
        "Use this tool for date/time questions."
    )
)

Let's test the agent.

In [12]:
await run_async("What’s the date today?")

"\n\nToday's date is June 9, 2026."

Well done on building an agent with access to real-time data. 



<a id="step5"></a>

## Step 5: Replace Your Date/time Tool with a MCP server

Now that we learned how to create a custom tool and provide the agent access to this tool. Let's now explore a trendy topic of [Model Context Protocol](https://modelcontextprotocol.io/introduction). We are going to explore how we can replace our custom tool with a simple MCP server that can serve our agent and provide similar information.

**Why MCP?** MCP servers provide:
- ✅ Standardized API interfaces
- 🔄 Reusable across projects
- 📦 Pre-built functionality

Let's replace our custom time tool with an official MCP time server:

### Installing Time MCP Server

We are going to start by installing this MCP server:


In [15]:
!python3 -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.2 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1


In [13]:
!pip install -q mcp-server-time


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


Now let's define our time_server:

In [16]:
from pydantic_ai.mcp import MCPServerStdio

time_server = MCPServerStdio(
    "python",
    args=[
        "-m", "mcp_server_time",
        "--local-timezone=America/New_York",
    ],
)

/tmp/ipykernel_105/3040644900.py:3: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  time_server = MCPServerStdio(


Finally, let's modify our agent to remove our previously defined tool, and add this MCP server instead.

In [17]:
agent = Agent(
    model=agent_model,
    mcp_servers=[time_server],
    system_prompt = (
        "You are a helpful agent and you have access to this tool:\n"
        "   get_current_time(params: dict)\n"
        "When the user asks for the current date or time, call get_current_time.\n"
    )
)

Great, let's see if the agent can use the MCP to give us the correct time now.

In [18]:
await run_async("What’s the date today?")

'\n\nThe current date is **Tuesday, June 9, 2026**. Let me know if you need further details!'


Tadaa! Now you have officially used an MCP server to power-up your agent. In the next section we show how you can your turn many ideas into real working projects by using 100s of free or paid MCP servers available today.



<a id="step6"></a>

## Step 6: Turn your agent to an airbnb finder

As we experience in the last section, MCP servers are really easy to use and they provide a standard way of providing LLMs the tools we need. There are already thousands of MCP servers available for us to use. There are some MCP trackers that you can always use to find out about available servers. Here are some for your reference:
- https://github.com/modelcontextprotocol/servers
- https://mcp.so/

We are going to use npx to launch out next server. Therefore, let's install the required dependencies.

In [19]:
# Install Node.js 20 via NodeSource
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!apt install -y nodejs

2026-06-09 08:15:40 - Installing pre-requisites
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy InRelease [3185 B]   
Get:3 https://repo.radeon.com/rocm/apt/7.2.1 jammy InRelease [2605 B]          
Get:4 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4006 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]                
Get:6 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1301 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7183 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]     
Get:11 https://repo.radeon.com/amdgpu/30.30.1/ubuntu jammy/main amd64 Packages [1332 B]m
Get:12 https://repo.radeon.com/ro

Verify `npm` and `npx` installation:

In [20]:
!node -v && npm -v && npx --version

v20.20.2
10.8.2
10.8.2




In this part of the workshop we are going to build an agent that can help you browse available Airbnbs to book. We can now build on top of what we have so far and add an open-source Airbnb MCP server to our agent. To do so, let's start by defining our Airbnb server.

In [21]:
airbnb_server = MCPServerStdio(
    "npx", args=["-y", "@openbnb/mcp-server-airbnb", "--ignore-robots-txt"]
)

/tmp/ipykernel_105/1537382601.py:1: DeprecationWarning: `MCPServerStdio` is deprecated and will be removed in v2. Use `MCPToolset('path/to/script.py')` for Python scripts, `MCPToolset('script.js')` for Node scripts, or `MCPToolset(fastmcp.client.transports.StdioTransport(command='...', args=[...]))` for arbitrary commands.
  airbnb_server = MCPServerStdio(


Let's update our agent.

In [22]:
system_prompt = """
You have access to three tools:
1. get_current_time(params: dict)
2. airbnb_search(params: dict)
3. airbnb_listing_details(params: dict)
When the user asks for listings, first call get_current_time, then airbnb_search, etc.
"""

agent = Agent(
    model=agent_model,
    mcp_servers=[time_server, airbnb_server],
    system_prompt=system_prompt,
)


Finally, let's try our agent and see if it can browse through Airbnb listings.

In [23]:
await run_async("Find a place to stay in Hyderabad for next Sunday for 3 nights for 2 adults?")

'\n\nHere are some Airbnb listings in Hyderabad for your stay from **June 14 to June 17, 2026** (3 nights) for 2 adults:\n\n### 🏠 Top Options:\n1. **The Aurelia: 3 BHK @ Banjara Hills**  \n   - **Price**: $256 for 3 nights  \n   - **Details**: 3 bedrooms, 2 baths, Guest Favorite  \n   - [View Listing](https://www.airbnb.com/rooms/1111546429562969055)\n\n2. **Deluxe Room - Starlight by Avana**  \n   - **Price**: $91 for 3 nights  \n   - **Details**: 1 bedroom, 1 king bed, 1 bath  \n   - [View Listing](https://www.airbnb.com/rooms/1462799445837907650)\n\n3. **Gachibowli Pent-House of Color’s**  \n   - **Price**: $146 for 3 nights  \n   - **Details**: 1 bedroom, 1 bed, 1 bath, Guest Favorite  \n   - [View Listing](https://www.airbnb.com/rooms/1429585002330236564)\n\n4. **New Premium 2BHK with Balcony**  \n   - **Price**: $131 for 3 nights  \n   - **Details**: 2 bedrooms, 2 baths, Superhost  \n   - [View Listing](https://www.airbnb.com/rooms/1665084026576926509)\n\n5. **SRT-302-Kondapur - 

In [53]:
await run_async("I want to visit Goa next week. What are the best dates?")

[06/09/26 08:42:37] INFO     HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 ]8;id=13137047;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=13137048;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

[06/09/26 08:42:43] INFO     HTTP Request: POST http://localhost:8000/v1/chat/completions "HTTP/1.1 ]8;id=13137053;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=13137054;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

"\n\nThe weather forecast for Goa next week shows persistent rain throughout the period, with high chances of precipitation (73-83%) and cloudy conditions. Here's the analysis:\n\n### **Best Dates for Your Visit:**\n- **June 10th (Tomorrow):**  \n  - **Morning (00:00–06:00):** Partly cloudy with a **10% chance of rain**. Temperatures range from **25–26°C**.  \n  - **Afternoon (12:00–18:00):** Light rain showers (78% chance). Temperatures rise to **28–29°C**.  \n  - **Evening (18:00–24:00):** Rain continues (83% chance).  \n\n- **June 11th:**  \n  - **Morning (00:00–06:00):** Light rain (77% chance).  \n  - **Afternoon (12:00–18:00):** Patchy rain (73% chance).  \n\n### **Why These Dates?**  \n- **June 10th morning** has the lowest rain chance (10%) and milder temperatures, making it the most favorable window.  \n- Avoid **June 9th** and **June 11th** due to heavy rain showers and high humidity.  \n\n### **Recommendation:**  \nIf you’re flexible, consider postponing your trip to **July 



<a id="step7"></a>

## Step 7: Challenge - Expand the Agent

**Task:** Add weather integration from any available MCP server you can find.

1. Launch weather MCP server
2. Add to agent's tools
3. Make agent suggest best travel dates based on weather

Happy coding! If you encounter issues or have questions, don’t hesitate to ask or raise an issue on our [Github page](https://github.com/ROCm/gpuaidev)!

In [34]:
!pip install mcp requests

In [38]:
!python weather_server.py

In [45]:
!pip install "pydantic-ai-slim[mcp]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 749.2/749.2 kB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.8 MB/s  0:00:00
  Attempting uninstall: starlette━━━━━━━━━━━━━━━━━━━━━━━━  5/13 [beartype]
    Found existing installation: starlette 0.52.1━━━━━━━━━━━━━  5/13 [beartype]
    Uninstalling starlette-0.52.1:━━━━━━━━━━━━━━━━━━━━━━━━  5/13 [beartype]
      Successfully uninstalled starlette-0.52.1━━━━━━━━━━━━━━━━━━━  6/13 [starlette]
  Attempting uninstall: keyring0m╸━━━━━━━━━━━━━━━━━━  7/13 [py-key-value-aio]
    Found existing installation: keyring 23.5.0━━━━━━━━━━━━━━━  7/13 [py-key-value-aio]
    Uninstalling keyring-23.5.0:1m╸━━━━━━━━━━━━━━━━━━  7/13 [py-key-value-aio]
      Successfully uninstalled keyring-23.5.0━━━━━━━━━━━━━━━━━  7/13 [py-key-value-aio]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [authlib]2/13 [authlib]slim]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviou

In [48]:
!pip install fastmcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [fastmcp]m8/9 [fastmcp]]


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset

weather_tools = MCPToolset("weather_server.py")

agent = Agent(
    model=agent_model,
    toolsets=[weather_tools]
)

In [42]:
agent = Agent(
    model=agent_model,
    mcp_servers=[weather_server],
    system_prompt="""
You are a travel planning assistant.

When a user asks for travel recommendations:

1. Use weather tools.
2. Analyze forecast conditions.
3. Recommend best travel dates.
4. Explain your reasoning.
"""
)

In [ ]:
result = agent.run_sync(
    "I want to visit Goa next week. What are the best dates?"
)

print(result.output)